# Capitan M11 — DA3 metric depth (the GPU step)

Runs **DepthAnything3 / DA3METRIC-LARGE** on wall photos and writes one metric-depth `.npz` per image to download. Everything else in M11 runs locally on CPU.

**First: Runtime → Change runtime type → GPU** (free T4 is enough).

Steps: install → upload an `images.zip` → run inference → download `depth.zip`. Then locally run `backend/scripts/build_depth_records.py --depth-dir <unzipped depth>`.

In [ ]:
# 1. Install DA3 + deps.
# --ignore-requires-python works around DA3's over-strict "requires-python <=3.13"
# (PEP 440 reads it as <=3.13.0, rejecting Colab's 3.13.x).
!pip -q install --ignore-requires-python "git+https://github.com/ByteDance-Seed/Depth-Anything-3.git" pillow numpy
# DA3 under-declares some runtime deps — install them explicitly.
!pip -q install addict omegaconf einops timm safetensors huggingface_hub

In [ ]:
# 2. Upload images.zip (your wall photos) and unzip into /content/images
import os, zipfile
from google.colab import files
up = files.upload()  # pick images.zip
os.makedirs('/content/images', exist_ok=True)
for name in up:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('/content/images')
print('unzipped into /content/images')

In [ ]:
# 3. Run DA3METRIC-LARGE -> metric-depth .npz per image
import os, glob, numpy as np, torch
from PIL import Image
from depth_anything_3.api import DepthAnything3

DIVISOR = 300.0      # DA3 metric formula: metric = focal_px * net_output / 300
SENSOR_35MM = 36.0
DEFAULT_F35 = 26.0   # fallback 35mm-equiv focal if no EXIF and no DA3 intrinsics
EXTS = ('.jpg', '.jpeg', '.png')

device = torch.device('cuda')
model = DepthAnything3.from_pretrained('depth-anything/DA3METRIC-LARGE').to(device)

def exif_focal_px(img, w, h):
    try:
        f35 = img.getexif().get(0xA405)  # FocalLengthIn35mmFilm
        if f35:
            return float(f35) / SENSOR_35MM * max(w, h), 'exif'
    except Exception:
        pass
    return DEFAULT_F35 / SENSOR_35MM * max(w, h), 'default'

os.makedirs('/content/depth', exist_ok=True)
paths = [p for p in glob.glob('/content/images/**/*', recursive=True) if p.lower().endswith(EXTS)]
print(len(paths), 'images')
for i, p in enumerate(paths):
    img = Image.open(p).convert('RGB')
    w, h = img.size
    pred = model.inference([p])
    depth_raw = np.asarray(pred.depth[0], dtype=np.float32)   # net output (H, W)
    # Focal in pixels: prefer DA3's estimated intrinsics, else EXIF, else default.
    focal_px, src = None, None
    try:
        K = np.asarray(pred.intrinsics[0])
        focal_px, src = float((K[0, 0] + K[1, 1]) / 2), 'da3'
    except Exception:
        focal_px = None
    if not focal_px or focal_px <= 0:
        focal_px, src = exif_focal_px(img, w, h)
    depth_m = depth_raw * focal_px / DIVISOR
    stem = os.path.splitext(os.path.basename(p))[0]
    np.savez_compressed(f'/content/depth/{stem}.npz', depth_m=depth_m, scan_id=stem,
                        model_version='da3metric-large', focal_px=float(focal_px), focal_source=src)
    if (i + 1) % 5 == 0 or i == len(paths) - 1:
        print(f'[{i+1}/{len(paths)}] {stem} focal={focal_px:.0f}({src}) '
              f'{depth_m.min():.2f}-{depth_m.max():.2f} m', flush=True)
print('done ->', len(os.listdir('/content/depth')), 'npz files')

In [ ]:
# 4. Zip + download the depth maps
!cd /content && zip -qr depth.zip depth
from google.colab import files
files.download('/content/depth.zip')